# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [163]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [164]:
# TODO: Import the necessary libs
# For example: 

import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser


In [165]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [166]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry.
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

# 1. Connect to the game database
chroma_client = chromadb.PersistentClient(path="chromadb")

# 2. Choose the embedding model
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)

# 3. Open the collection containing game information
collection = chroma_client.get_collection(
    name="udaplay_games",
    embedding_function=embedding_fn
)

# 4. Create a tool to search for games
@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds most similar results in the vector DB

    Args:
        query: a question about game industry

    Returns:
        Results as a list. Each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """

    # Search the database for the 5 most relevant games
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    # Prepare the search results
    games = []

    if results["metadatas"] and results["metadatas"][0]:
        for game in results["metadatas"][0]:
            games.append({
                "Platform": game.get("Platform", "Unknown"),
                "Name": game.get("Name", "Unknown"),
                "YearOfRelease": game.get("YearOfRelease", "Unknown"),
                "Description": game.get(
                    "Description", "No description available"
                )
            })

    # Return the results in a readable format
    return json.dumps(games, indent=2)

#### Evaluate Retrieval Tool

In [167]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents,
#    it will analyze the usability of the documents to respond to that question.
#    args:
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

# 1. Define EvaluationReport model for structured output
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Are the documents useful?")
    description: str = Field(description="Why are they useful or not?")

# 2. Create the evaluation tool
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
        """
        Based on the user's question and on the list of retrieved documents,
        it will analyze the usability of the documents to respond to that question.

        Args:
            question: original question from user
            retrieved_docs: retrieved documents most similar to the user query in the Vector Database

        Returns:
            The result includes:
            - useful: whether the documents are useful to answer the question
            - description: description about the evaluation result
        """

        # Create the LLM judge
        llm_judge = LLM(
            model="gpt-4o-mini",
            temperature=0.0,
            api_key=OPENAI_API_KEY,
        )

        # Prepare the prompt
        prompt = f"""
        Your task is to evaluate whether the documents are enough to answer
        the user's question. Explain your decision.

        User question: {question}

        Retrieved documents: {retrieved_docs}
        """

        # Ask the LLM for an evaluation
        response = llm_judge.invoke(
            input=prompt,
            response_format=EvaluationReport
        )

        parser = PydanticOutputParser(model_class=EvaluationReport)
        evaluation = parser.parse(response)

        return json.dumps({
            "useful": evaluation.useful,
            "description": evaluation.description
        }, indent=2)



#### Game Web Search Tool

In [168]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.

# 1. Connect to Tavily
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


# 2. Create a tool to search the web
@tool
def game_web_search(question: str) -> str:
    """Search the web for information about video games."""

    # Search for relevant web pages
    response = tavily_client.search(
        query=question,
        max_results=5
    )

    # Collect the most useful information
    results = []

    for result in response.get("results", []):
        results.append({
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "content": result.get("content", "")
        })

    # Return the results in readable JSON format
    return json.dumps(results, indent=2)

### Agent

In [169]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
# Create the UdaPlay agent

udaplay_agent = Agent(
    model_name="gpt-4o-mini",

    instructions="""
    You are UdaPlay, an AI assistant that answers questions about video games.

    Follow these steps:
    1. Search the game database first.
    2. Check if the results answer the question.
    3. If the results are not enough, search the web.
    4. Give the user a clear and helpful answer.
    """,

    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ],

    temperature=0.7
)

# Confirm that the agent was created
print("UdaPlay Agent created successfully!")
print("Model: gpt-4o-mini")
print(f"Tools available: {len(udaplay_agent.tools)}")

UdaPlay Agent created successfully!
Model: gpt-4o-mini
Tools available: 3


In [170]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

# Test the agent with sample questions
queries = [
    "When were Pokémon Gold and Silver released?",
    "Which Mario game was the first 3D platformer?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

# Use the same session for all questions
session_id = "demo_session"

for i, query in enumerate(queries, 1):
    print(f"\n{'=' * 50}")
    print(f"Question {i}: {query}")
    print("=" * 50)

    # Ask the agent
    run = udaplay_agent.invoke(query, session_id=session_id)

    # Get the final answer
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    final_answer = None

    for message in reversed(messages):
        if isinstance(message, AIMessage) and message.content:
            if not getattr(message, "tool_calls", None):
                final_answer = message.content
                break

    # Display the answer
    if final_answer:
        print(final_answer)
    else:
        print("No final answer was found.")

print("\nAll questions have been processed.")


Question 1: When were Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are notable for being the second generation of Pokémon, introducing new regions, Pokémon, and gameplay mechanics.

Question 2: Which Mario game was the first 3D platformer?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termin

### (Optional) Advanced

In [171]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

### TESTING MODULES

In [183]:
print("Module Tests")
print('-------------------------------------------')
# Testing Database game if database contains any data
print("Testing Database game if database contains any data")

if collection.count() > 0:
    print("✓ PASS: Game database contains data.")
else:
    print("x FAIL: Game database is empty.")
print('-------------------------------------------')

# Testing if the tool for searching the game can retrieve a name of the game
print("Testing game retrieval tool")

result = retrieve_game("Mario games")

if result:
    print("✓ PASS: Retrieval tool returned a result.")
else:
    print("x FAIL: Retrieval tool returned nothing.")
print('-------------------------------------------')

print("Testing UdaPlay agent")
if callable(udaplay_agent.invoke):
    print("✓ PASS: Agent is ready to be called.")
else:
    print("x FAIL: Agent is not ready.")
print('-------------------------------------------')

print("Testing UdaPlay agent response")
run = udaplay_agent.invoke("Recommend a Mario game.", session_id="test")
messages = run.get_final_state().get("messages", [])
answer = ""

for message in reversed(messages):
    if isinstance(message, AIMessage) and message.content and not getattr(message, "tool_calls", None):
        answer = message.content
        break

if answer:
    print("✓ PASS: Final answer received.")
    print(answer)
else:
    print("x FAIL: No final answer generated.")
print('-------------------------------------------')

Module Tests
-------------------------------------------
Testing Database game if database contains any data
✓ PASS: Game database contains data.
-------------------------------------------
Testing game retrieval tool
✓ PASS: Retrieval tool returned a result.
-------------------------------------------
Testing UdaPlay agent
✓ PASS: Agent is ready to be called.
-------------------------------------------
Testing UdaPlay agent response
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
✓ PASS: Final answer received.
I recommend **Mario Kart 8 Deluxe** for the Nintendo Switch, released in 2017. This game is an enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gamepla